# Phase 2: Classification & Clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/forest-carbon-pipeline/blob/main/notebooks/02_Classification_Clustering.ipynb)

Unsupervised K-Means clustering to isolate distinct spectral types.


In [ ]:
# Setup: locate the repo's `src` package (any cwd inside the repo, or fresh clone)
import os
import subprocess
import sys

REPO = "forest-carbon-pipeline"
if os.path.isdir("src"):
    ROOT = os.getcwd()
elif os.path.isdir(os.path.join("..", "src")):
    ROOT = os.path.abspath("..")
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline.git"],
                   check=False)
    ROOT = os.path.abspath(REPO)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
if not os.path.isdir(os.path.join(ROOT, "src")):
    raise RuntimeError(
        f"Could not locate the pipeline source at {ROOT}. Open this notebook "
        "from a repo checkout, or check your network so the bootstrap can "
        "clone https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline"
    )

from src.phase1_demarcate import demarcate_forest_pixels  # noqa: E402
QUARANTINE = os.path.join(ROOT, "quarantine")
print(f"Pipeline package ready (ROOT = {ROOT})")


## Synthetic scene

Regenerated deterministically so this notebook runs standalone.


In [ ]:
# Synthetic Sentinel-2-like scene: 100 x 100 px at 20 m (deterministic)
import numpy as np

np.random.seed(42)
nir_band = np.random.uniform(0.2, 0.8, (100, 100))
red_band = np.random.uniform(0.05, 0.3, (100, 100))
savi = ((nir_band - red_band) / (nir_band + red_band + 0.5)) * 1.5  # L = 0.5
print("Scene ready: 100x100 px @ 20 m, seed=42")

forest_mask, ndvi, input_hash = demarcate_forest_pixels(
    nir_band, red_band, quarantine_dir=QUARANTINE)


## From clusters to density classes

K-Means groups forest pixels into 3 spectral TYPEs. Separately, an NDVI-derived canopy-cover proxy is cut into the FSI density legend: VDF >= 70%, MDF 40-70%, OF 10-40%. Keeping TYPE and DENSITY as separate layers is what makes the Phase 3 matrix overlay possible.


In [ ]:
from src.phase2_preprocess import classify_spectral_types

type_map, density_map = classify_spectral_types(ndvi, savi, forest_mask)

vals, counts = np.unique(type_map[type_map > 0], return_counts=True)
print("Spectral type shares:", {int(v): f"{c / counts.sum() * 100:.1f}%" for v, c in zip(vals, counts)})
vals, counts = np.unique(density_map[density_map > 0], return_counts=True)
print("Density classes     :", {int(v): f"{c / counts.sum() * 100:.1f}%" for v, c in zip(vals, counts)})


Future redesign: replace K-Means with a supervised U-Net CNN with skip connections once labelled training tiles exist.
